# SmartMOM: Whisper Base/Turbo + NVIDIA Sortformer
Reproducible GPU evaluation on five AMI meetings. Use a GPU runtime. The notebook writes machine-readable results and never substitutes vendor benchmark numbers for project measurements.

In [ ]:
import os
import subprocess

assert subprocess.run(["nvidia-smi"], check=False).returncode == 0, (
    "Select a Colab GPU runtime first."
)
REPO_URL = "https://github.com/Kalsoomkhn/Smart-MOM-Bot.git"
REPO_DIR = "/content/smartmombot"
if not os.path.exists(REPO_DIR):
    assert REPO_URL.startswith(("https://", "git@")), "Set REPO_URL before continuing."
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!python -m pip install -q Cython packaging
!python -m pip install -q -r experiments/requirements-colab.txt

In [ ]:
import torch
import transformers

print(
    {
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "transformers": transformers.__version__,
    }
)

## Run Whisper Base and Turbo
Complete AMI recordings are scored with normalized WER. English is forced to prevent the catastrophic language-detection failure observed in the baseline.

In [ ]:
!python experiments/benchmark_stt_diarization.py --component stt --work-dir /content/smartmom-evaluation --output-dir /content/smartmom-stt-results

## Run Sortformer
The first 300 seconds of every meeting are scored with overlap included and a 0.25-second collar. You may need to authenticate with Hugging Face when prompted.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!python experiments/benchmark_stt_diarization.py --component diarization --diarization-seconds 300 --work-dir /content/smartmom-evaluation --output-dir /content/smartmom-diarization-results

In [ ]:
import pandas as pd

stt = pd.read_json("/content/smartmom-stt-results/results.json")
diar = pd.read_json("/content/smartmom-diarization-results/results.json")
display(stt)
display(diar)
display(
    stt.groupby("model", dropna=False)[["wer", "real_time_factor", "peak_gpu_memory_mb"]].mean()
)
display(
    diar.groupby("model", dropna=False)[["der", "real_time_factor", "peak_gpu_memory_mb"]].mean()
)

In [ ]:
!cd /content && zip -qr smartmom-results.zip smartmom-stt-results smartmom-diarization-results
from google.colab import files

files.download("/content/smartmom-results.zip")